# Video Deep Learning Features — SwinT / FabNet / CLIP

| Model | Wrapper | Output dim | Input size |
|---|---|---|---|
| Swin Transformer | `SwinTWrapper` | 768 | 224×224 |
| FAb-Net | `FabNetWrapper` | 256 | 112×112 |
| CLIP ViT-H/14 | `ClipWrapper` | 1024 | 224×224 |

Each wrapper handles preprocessing internally — pass a raw `(T, 3, H, W)` uint8 tensor and get a `(T, D)` float feature tensor back.
Demo uses `multispeaker_short.mp4` sampled at 5 fps.

In [1]:
from pathlib import Path

from exordium.video.core.io import load_video

video_path = Path("../tests/fixtures/video/multispeaker_short.mp4")

# Load frames as uint8 tensor (T, 3, H, W)
frames, fps = load_video(video_path, fps=5)
print(f"Loaded {frames.shape[0]} frames @ {fps} fps")
print(f"Shape : {tuple(frames.shape)}  dtype={frames.dtype}")

/Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 5 frames @ 5 fps
Shape : (5, 3, 720, 1280)  dtype=torch.uint8


objc[76484]: Class AVFFrameReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x11f7f03a8) and /opt/homebrew/Cellar/ffmpeg/8.0.1/lib/libavdevice.62.1.100.dylib (0x334e98328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[76484]: Class AVFAudioReceiver is implemented in both /Users/fodorad/miniconda3/envs/exordium/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x11f7f03f8) and /opt/homebrew/Cellar/ffmpeg/8.0.1/lib/libavdevice.62.1.100.dylib (0x334e98378). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


## 1. Swin Transformer

In [2]:
from exordium.video.deep import SwinTWrapper

swin = SwinTWrapper(device_id=-1)  # -1 → CPU
swin_feats = swin(frames)  # (T, 768) Tensor

print(f"Output: {tuple(swin_feats.shape)}  dtype={swin_feats.dtype}")

Output: (5, 768)  dtype=torch.float32


## 2. FAb-Net

Pre-trained on face sequences for identity-invariant appearance features.
Expects face crops; here we feed full frames for illustration.

In [3]:
from exordium.video.deep import FabNetWrapper

fabnet = FabNetWrapper(device_id=-1)
fabnet_feats = fabnet(frames)  # (T, 256) Tensor

print(f"Output: {tuple(fabnet_feats.shape)}  dtype={fabnet_feats.dtype}")

2026-03-23 11:43:42 INFO FAb-Net is loaded to cpu.


Output: (5, 256)  dtype=torch.float32


## 3. CLIP

In [4]:
from exordium.video.deep import ClipWrapper

clip = ClipWrapper(device_id=-1)
clip_feats = clip(frames)  # (T, 1024) Tensor

print(f"Output: {tuple(clip_feats.shape)}  dtype={clip_feats.dtype}")

Output: (5, 1024)  dtype=torch.float32


## 4. Feature Summary

In [5]:
print(f"Frames: {frames.shape[0]}  ({fps} fps)")
print()
print(f"{'Model':<22} {'Output shape':<20} {'Dim':>6}")
print("-" * 50)
for name, feats in [
    ("Swin Transformer", swin_feats),
    ("FabNet", fabnet_feats),
    ("CLIP ViT-H/14", clip_feats),
]:
    print(f"{name:<22} {str(tuple(feats.shape)):<20} {feats.shape[-1]:>6}")

Frames: 5  (5 fps)

Model                  Output shape            Dim
--------------------------------------------------
Swin Transformer       (5, 768)                768
FabNet                 (5, 256)                256
CLIP ViT-H/14          (5, 1024)              1024
